# sattebazi: all approaches on Kaggle (2x T4)

**Inputs to attach** (Add Input): the code dataset (this repo as a zip) and the data dataset (`fantasy_data.zip`, i.e. `data/raw`).
Optional: the hidden `test_fixtures.json` once released; a previous version's output of this notebook to resume.

**Settings**: Accelerator = GPU T4 x2. Internet not needed. Run with **Save Version -> Save & Run All (Commit)** so it survives closing the browser.

**Resume**: every stage writes `results/stages/<stage>.json` when it finishes and is skipped next time; every CB run checkpoints each epoch
(`model_artifacts/checkpoints/<run>/last.pt`) and resumes. **To skip preprocessing**: attach `fantasy_processed.zip` (built locally: processed data + finished stages); after a crash, attach this notebook's last output instead. `RESUME_FROM = "auto"` picks whichever has more finished stages.

Fresh run: about 3.8 h for B0s + CB, then about 3 h for ADV (+1.5 h if ADV wins and its FINAL runs are trained).
**To add ADV to the finished run**: attach the previous version's output as an input; every finished stage is skipped and only the ADV stages run.

In [ ]:
# 1. setup: locate inputs, copy code, link raw data, resume from a previous output if attached
import glob, json, os, shutil, subprocess, zipfile
WORK = "/kaggle/working/sattebazi"
RESUME_FROM = "auto"   # "auto" | None (fresh start) | a folder that contains results/stages/
TEST_FILE = None       # hidden test file path; None = auto-detect test_fixtures.json / test.json under /kaggle/input

def find(name, skip=("/sattebazi/",)):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return sorted(h for h in hits if not any(s in h for s in skip))

code_root = os.path.dirname(os.path.dirname(find("src/run_kaggle.py")[0]))
print("code:", code_root)
shutil.copytree(code_root, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns("raw"))

if RESUME_FROM == "auto":  # a previous run's output, or the processed-data zip (fantasy_processed.zip): whichever has more stages
    marks = glob.glob("/kaggle/input/**/results/stages/*.json", recursive=True)
    roots = {m.rsplit("/results/stages/", 1)[0] for m in marks if not m.startswith(code_root)}
    RESUME_FROM = max(roots, key=lambda r: len(glob.glob(f"{r}/results/stages/*.json"))) if roots else None
if RESUME_FROM and not os.path.isdir(f"{WORK}/results/stages"):
    print("resuming from", RESUME_FROM, "| stages done:", sorted(os.listdir(f"{RESUME_FROM}/results/stages")))
    for d in ("results", "model_artifacts", "data/interim", "data/processed", "data/audit"):
        if os.path.isdir(f"{RESUME_FROM}/{d}"):
            shutil.copytree(f"{RESUME_FROM}/{d}", f"{WORK}/{d}", dirs_exist_ok=True)

people = find("registry/people.csv")
if not people:  # dataset uploaded as a zip that Kaggle did not extract
    for z in find("*.zip"):
        with zipfile.ZipFile(z) as f:
            if any(n.endswith("registry/people.csv") for n in f.namelist()):
                f.extractall("/kaggle/tmp_data"); print("unzipped", z)
    people = sorted(glob.glob("/kaggle/tmp_data/**/registry/people.csv", recursive=True))
raw = os.path.dirname(os.path.dirname(people[0]))
assert os.path.isdir(f"{raw}/cricsheet/male"), f"expected {raw}/cricsheet/male/<year>/*.json"
os.environ["RAW_DIR"] = raw  # read in place (read-only input), never copied into /kaggle/working
os.chdir(WORK)
n = len(glob.glob(f"{raw}/cricsheet/*/*/*.json"))
print("raw:", raw, "|", n, "match files")
if TEST_FILE:
    os.environ["TEST_FILE"] = TEST_FILE
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -c "import torch, lightgbm, scipy, pandas; print('torch', torch.__version__, 'cuda', torch.cuda.device_count(), '| lightgbm', lightgbm.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)"

In [ ]:
# 2. data pipeline + gates G1-G6 (ingest, Φ, roles, pseudo-squads, features; Φ / causality / selection tests)
!python src/run_kaggle.py --stage pipeline

In [ ]:
# 3. staleness-aware features for B0s + gate G7 (staleness time-travel test)
!python src/run_kaggle.py --stage stale

In [ ]:
# 4. B0s: F24 and F25 folds (predictions on the Δ grid) + the FINAL B0s model (the fallback submission)
!python src/run_kaggle.py --stage b0s

In [ ]:
# 5. CB gates and mechanics: T1/T1c/T4/T7, smoke (1 epoch per arm), T3 overfit, evaluate.py on the mock fixtures
!python src/run_kaggle.py --stage cb_checks

In [ ]:
# 6. the 12 CB runs (CB64/CB128 x F24/F25/FINAL x seeds 0,1), one queue per GPU, checkpoint every epoch
# per-run logs: results/logs/<run>.log ; status: results/logs/cb_status.json
!python src/run_kaggle.py --stage cb_runs

In [ ]:
# 7. evaluate once (SPEC §13.6) and apply the pre-registered rules (§13.7)
!python src/run_kaggle.py --stage report
print(open('results/experiments/E22_final/decision.md').read())

In [ ]:
# 7b. ADV gates and mechanics: T1 T2 T4 T6 T7 T11 T14 T16, smoke (every phase), T3 overfit, evaluate.py on the mock fixtures
!python src/run_kaggle.py --stage adv_checks

In [ ]:
# 7c. ADV runs: F25 and F24 x seeds 0,1 (phases P1 -> P2 -> P3 -> P4), one queue per GPU, checkpoint every epoch
# per-run logs: results/logs/ADV_<split>_s<seed>.log ; status: results/logs/adv_status.json
!python src/run_kaggle.py --stage adv_runs

In [ ]:
# 7d. ADV evaluation and the pre-registered rule (SPEC §14)
!python src/run_kaggle.py --stage adv_report
print(open('results/experiments/E23_adv/decision.md').read())

In [ ]:
# 7e. only if ADV won: the FINAL two-seed ADV runs
!python src/run_kaggle.py --stage adv_final

In [ ]:
# 8. submission model + CSVs (mock; hidden test file if attached) + T5 determinism
!python src/run_kaggle.py --stage submit

In [ ]:
# 9. bundle everything worth downloading into /kaggle/working/results_bundle.zip, then show a summary
!python src/run_kaggle.py --stage bundle
import pandas as pd
st = {f[:-5]: json.load(open(f"results/stages/{f}")).get("minutes") for f in sorted(os.listdir("results/stages"))}
print("minutes per stage:", st)
for f in ("results/hidden_predictions.csv", "results/mock_predictions.csv"):
    if os.path.exists(f):
        print(f); display(pd.read_csv(f).head(11))